# **ColabAlign**

## Fast pairwise protein secondary structure comparisons using multiprocessing

 [![Code DOI](https://img.shields.io/badge/DOI-10.5281/zenodo.14169501-blue)](https://doi.org/10.5281/zenodo.14169501) [![Paper DOI](https://img.shields.io/badge/DOI-10.1101/2025.10.06.677802-green)](https://doi.org/10.1101/2025.10.06.677802)

---

Create a phylogenetic tree that compares the secondary structure of proteins, rather than nucleotide or amino acid sequence. Scoring uses the [US-align](https://zhanggroup.org/US-align/) algorithm by [Zhang _et al._, (2022)](https://doi.org/10.1038/s41592-022-01585-1).

A score of **<0.17** indicates similarity indistinguishable from a random pair of structures, where as as score **≥0.50** indicates a pair with broadly the same fold ([Xu _et al._, 2010](https://doi.org/10.1093/bioinformatics/btq066))

---

## **Usage**

1) **Click `Runtime` -> `Change runtime type` -> `v5e-1 TPU`**

2) **Either upload `.pdb` and/or `.cif` format files directly or compressed into a `.zip` or tar (`.tar`, `.tar.gz`/`.tgz`, `.tar.bz2`, `.tar.xz`) file to the Colab instance by clicking the folder icon on the left, then dragging and dropping your structures.**

3) **Adjust the clustering threshold (horizontal bar) as needed.**

4) **Click `Run all`**

## N.B.

* Other runtimes can be used if you are paying for compute units. Ensure you select a runtime with more than 2 CPU cores, particularly if you plan to analyse a large dataset.

* Structure files containing more than one chain are split into separate .pdb files by BeEM prior to alignment. By default, only the **first chain** is considered (*first chain only* mode). If you want to use all chains, change `mode` to *all chains*

* Every structure needs a unique name. If several files share a name (e.g. `ranked_0.pdb` from different AlphaFold runs, in separate folders or archives), the first keeps its name and the rest are aligned as `ranked_0_dup1`, `ranked_0_dup2`, and so on. Your files are not renamed. `duplicates.csv` in the downloaded results lists every clashing file, where it came from, and the name it was given.

* Structural dendrograms generated by ColabAlign should be analysed as unrooted trees.

* Clustering `threshold` is applied to distances in the UPGMA tree by TreeCluster's default `max_clade` method. It is not a hard minimum pairwise TM-score cutoff.


In [ ]:
# @title Set up ColabAlign conda environment and prepare file structure

%%bash
set -euo pipefail

# ColabAlign v1.2.0 - commit 158663d
COLABALIGN_SHA="158663d8c246747aa019ab3323abf10a39ee074c"

rm -rf sample_data ColabAlign_git
git init -q ColabAlign_git
git -C ColabAlign_git fetch -q --depth 1 https://github.com/crfield18/ColabAlign.git "$COLABALIGN_SHA"
git -c advice.detachedHead=false -C ColabAlign_git checkout -q FETCH_HEAD


test "$(git -C ColabAlign_git rev-parse HEAD)" = "$COLABALIGN_SHA"

pip install konda==0.1.0
konda config --file /usr/local/.condarc --remove channels defaults
konda config --file /usr/local/.condarc --add channels conda-forge

konda create --yes -n colabalign --file ColabAlign_git/locks/conda-linux-64.lock --quiet
konda activate colabalign
konda run "python -m pip install --no-deps --no-build-isolation --quiet ./ColabAlign_git"
rm -rf ColabAlign_git


In [ ]:
# @title Align Structures
# Run colabalign with all available cores

threshold = 0.20 # @param {type:"slider", min:0.01, max:1.00, step:0.01}
mode = 'first chain only' # @param ["first chain only", "all chains"]

if mode == 'all chains':
    mode = 'all'
elif mode == 'first chain only':
    mode = 'first'

import os
import shutil
from pathlib import Path

CONTENT = Path('/content')
MODELS = CONTENT / 'models'
INPUT_SUFFIXES = ('.pdb', '.cif', '.zip', '.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tar.xz')
# models/ holds staged inputs, results/ holds previous runs (never re-use them as
# inputs), drive/ is a mounted Google Drive (never move files out of it).
SKIP_TOP_LEVEL = {'models', 'results', 'drive', 'sample_data'}

MODELS.mkdir(exist_ok=True)
(CONTENT / 'results').mkdir(exist_ok=True)

# Move uploads into models/, keeping their folder layout so files with the same
# name in different folders don't overwrite each other. ColabAlign gives any
# name clashes _dup1, _dup2, ... names and lists them in duplicates.csv.
# .zip and tar archives are moved as-is; colabalign extracts them.
for root, dirs, files in os.walk(CONTENT):
    rel_root = Path(root).relative_to(CONTENT)
    dirs[:] = sorted(d for d in dirs if not d.startswith('.')
                     and not (rel_root == Path('.') and d in SKIP_TOP_LEVEL))
    for name in sorted(files):
        lower = name.lower()
        if name.startswith('.') or not lower.endswith(INPUT_SUFFIXES):
            continue
        if rel_root == Path('.') and lower.startswith('colabalign_results_'):
            continue  # a previous results download
        dest = MODELS / rel_root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(Path(root) / name), str(dest))

# colabalign reads each input directory's own contents, so pass every folder
# under models/ that holds an input file.
input_dirs = sorted({
    root for root, _, files in os.walk(MODELS)
    if any(f.lower().endswith(INPUT_SUFFIXES) and not f.startswith('.') for f in files)
})
if not input_dirs:
    raise SystemExit('No .pdb, .cif, .zip or tar files found. Upload your structures first.')

# colabalign lives in the konda env (its own Python), not Colab's, so import
# it from a short script run by the env's Python. Colab's packages are left
# untouched and no runtime restart is needed.
cores = os.cpu_count()
run_script = Path('/tmp/colabalign_run.py')
run_script.write_text(f'''from colabalign import ColabAlign

if __name__ == '__main__':
    print('Starting alignment with {cores} cores...')
    ColabAlign(
        inputs={input_dirs!r},
        output={str(CONTENT / 'results')!r},
        cores={cores},
        threshold={threshold!r},
        mode={mode!r},
    ).run()
''')

!konda run "python -u {run_script}"


In [ ]:
# @title Zip and download results

import os
import datetime
from google.colab import files

# Name the zipped results file using the current date and time
# to not accidentally overwrite older results files when downloading
current_dt = datetime.datetime.now()
zip_filename = f'colabalign_results_{current_dt.strftime("%Y%m%d-%H%M")}.zip'

# Using the built-in zip function rather than a python module for improved efficiency
os.system(f'zip -r {zip_filename} results -x "*.pkl"')
files.download(zip_filename)
